# Spam Email Detection with NLP
## Notebook 3 — Text Preprocessing and Feature Engineering
**Author: Tinashe Clifford Makandise**

### Purpose
Prepare reproducible TF-IDF pipelines for later model comparisons. Learn text cleaning, tokenization, stopwords, stemming and sparse features. No model is selected or evaluated here. The test CSV remains unopened.

Run Notebooks 1 and 2 first. Save this file in `spam-email-detector/notebooks/`. Use the same Python environment; the required packages are pandas, numpy and scikit-learn. Run cells from top to bottom. All text examples below are invented, not excerpts from the dataset.

In [1]:
from pathlib import Path
import hashlib, json, re, unicodedata
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REPORTS = ROOT / "reports"
manifest = json.loads((REPORTS / "data_manifest.json").read_text(encoding="utf-8"))
train_path = ROOT / "data/processed/train.csv"
train_hash = hashlib.sha256(train_path.read_bytes()).hexdigest()
assert train_hash == manifest["train_file_sha256"], "Rerun Notebook 1 to regenerate consistent files."
train = pd.read_csv(train_path, keep_default_na=False)
assert len(train) == manifest["train_rows"]
assert set(train.row_id) == set(manifest["train_row_ids"])
assert set(train.group_id) == set(manifest["train_groups"])
assert set(train.group_id).isdisjoint(manifest["test_groups"])
assert train.spam.isin([0, 1]).all() and train.text.str.strip().ne("").all()
X = train["text"]
y = train["spam"]
groups = train["group_id"]
print(f"Verified {len(X):,} training emails. Test file not opened.")

Verified 4,547 training emails. Test file not opened.


### 1. Cleaning is a modelling choice
Aggressive cleaning can discard useful signals: exclamation marks, digits, URLs and negation may help distinguish messages. Our initial candidate will use scikit-learn's built-in lowercase conversion and word tokenization, with no stemming or stopword removal. We will compare a stopword-removal candidate later using training-only cross-validation.

The following function is an **educational demonstration only**. It standardises Unicode and whitespace. It does not modify the stored dataset or become a custom callable in the saved model. Keeping production pipelines built from standard scikit-learn components simplifies deployment.

In [2]:
def clean_demo(text):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", text)).strip().lower()

example = "  Congratulations!!! YOU won 500 dollars. Click now!  "
cleaned = clean_demo(example)
print("Invented input:", example)
print("Whitespace/lowercase cleanup:", cleaned)
words = re.findall(r"(?u)\b\w\w+\b", cleaned)
print("Word tokens:", words)
print("After English stopword removal:", [w for w in words if w not in ENGLISH_STOP_WORDS])
print("Notice: tokenization already omitted punctuation and one-character tokens.")

Invented input:   Congratulations!!! YOU won 500 dollars. Click now!  
Whitespace/lowercase cleanup: congratulations!!! you won 500 dollars. click now!
Word tokens: ['congratulations', 'you', 'won', '500', 'dollars', 'click', 'now']
After English stopword removal: ['congratulations', 'won', '500', 'dollars', 'click']
Notice: tokenization already omitted punctuation and one-character tokens.


### 2. Stopwords and stemming
Stopwords are frequent words such as “the” and “is”. Removing them may reduce vocabulary size, but may also remove useful phrases or negation. Stemming strips suffixes using rules; it does not always produce a real word and does not reliably map “ran” to “run”. Lemmatization is a different linguistic process.

For a brief optional stemming demonstration, install NLTK with `%pip install nltk` if desired. PorterStemmer needs no downloaded corpora. NLTK is **not required** for the pipelines below. We leave stemming out of the first modelling experiment to keep the baseline interpretable and deployment straightforward.

In [3]:
try:
    from nltk.stem import PorterStemmer
except ImportError:
    print("Optional stemming demo skipped: NLTK is not installed.")
else:
    stemmer = PorterStemmer()
    for word in ["playing", "played", "running", "runs", "ran"]:
        print(f"{word:10s} -> {stemmer.stem(word)}")

Optional stemming demo skipped: NLTK is not installed.


### 3. Understand TF-IDF on invented documents
Term frequency measures occurrence within a document. Inverse document frequency downweights terms appearing in many training documents. With the defaults used here, smoothed IDF is `log((1 + n_documents)/(1 + document_frequency)) + 1`; sublinear term frequency uses `1 + log(count)` for nonzero counts, followed by L2 row normalisation.

TF-IDF does not know the labels and does not inherently assign a high weight to “spam words”. A classifier later learns associations between these numerical features and labels. Bigrams retain adjacent word pairs. These matrices are sparse: only nonzero values are stored.

In [4]:
toy_text = ["team meeting today", "free prize today", "team report meeting"]
toy = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
toy_matrix = toy.fit_transform(toy_text)
# Dense conversion is safe only for this tiny invented example.
print(pd.DataFrame(toy_matrix.toarray(), columns=toy.get_feature_names_out()).round(3).to_string(index=False))
print("Shape:", toy_matrix.shape, "Nonzero entries:", toy_matrix.nnz)
assert (toy_matrix.data >= 0).all()

 free  free prize  meeting  meeting today  prize  prize today  report  report meeting  team  team meeting  team report  today
0.000       0.000    0.394          0.517  0.000        0.000    0.00            0.00 0.394         0.517         0.00  0.394
0.467       0.467    0.000          0.000  0.467        0.467    0.00            0.00 0.000         0.000         0.00  0.355
0.000       0.000    0.373          0.000  0.000        0.000    0.49            0.49 0.373         0.000         0.49  0.000
Shape: (3, 12) Nonzero entries: 15


### 4. Freeze candidate definitions before comparing models
We define two candidate representations: word unigrams/bigrams with and without built-in English stopwords. Both cap the vocabulary at 30,000 features, require a term to appear in at least two fitting documents, and use sublinear TF. These are initial experiment settings, not proven best values. A vocabulary cap controls memory but may discard informative terms.

We preserve the supplied training strings. The corpus is historical and source-specific; names and organisation terms can become shortcuts. Future performance on modern inboxes needs separate evidence. Neither representation is a phishing detector or a safety guarantee.

In [5]:
VECTORIZER_SETTINGS = {
    "lowercase": True,
    "ngram_range": (1, 2),
    "min_df": 2,
    "max_features": 30000,
    "sublinear_tf": True,
    "norm": "l2",
}
CANDIDATES = {"word_tfidf": None, "word_tfidf_stopwords": "english"}

def make_vectorizer(candidate="word_tfidf"):
    if candidate not in CANDIDATES:
        raise ValueError(f"Unknown candidate: {candidate}")
    return TfidfVectorizer(**VECTORIZER_SETTINGS, stop_words=CANDIDATES[candidate])

def make_pipeline(model_name="naive_bayes", candidate="word_tfidf"):
    if model_name == "naive_bayes":
        classifier = MultinomialNB(alpha=1.0)
    elif model_name == "logistic_regression":
        classifier = LogisticRegression(C=1.0, max_iter=2000, random_state=42)
    else:
        raise ValueError(f"Unknown model: {model_name}")
    return Pipeline([("tfidf", make_vectorizer(candidate)), ("classifier", classifier)])

example_pipeline = make_pipeline()
assert not hasattr(example_pipeline.named_steps["tfidf"], "vocabulary_")
print(example_pipeline)
print("Pipeline created but not fitted. Each call creates fresh components.")

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=30000, min_df=2,
                                 ngram_range=(1, 2), sublinear_tf=True)),
                ('classifier', MultinomialNB())])
Pipeline created but not fitted. Each call creates fresh components.


### 5. Group-aware validation and leakage prevention
Normalised case/whitespace variants must remain in the same fold. StratifiedGroupKFold approximately balances class proportions while keeping groups intact. It does not detect all near-duplicates or related campaigns.

In Notebook 4, pass the **entire pipeline**, original strings and groups into the cross-validation search. Never fit TF-IDF on all training messages before cross-validation: that would expose validation vocabulary and IDF statistics to the fitting process. The final held-out test set is reserved for Notebook 5.

In [6]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(cv.split(X, y, groups=groups))
fold_summary = []
for fold_number, (fit_idx, valid_idx) in enumerate(folds, 1):
    assert set(groups.iloc[fit_idx]).isdisjoint(set(groups.iloc[valid_idx]))
    assert set(y.iloc[fit_idx]) == {0, 1} and set(y.iloc[valid_idx]) == {0, 1}
    fold_summary.append({"fold": fold_number, "fit_rows": len(fit_idx),
                         "validation_rows": len(valid_idx),
                         "validation_spam_percent": round(100 * y.iloc[valid_idx].mean(), 2)})
assert sorted(np.concatenate([v for _, v in folds]).tolist()) == list(range(len(X)))
print(pd.DataFrame(fold_summary).to_string(index=False))
print("Every row validated once; no normalised-group overlap within any fold.")

 fold  fit_rows  validation_rows  validation_spam_percent
    1      3637              910                    24.07
    2      3638              909                    23.98
    3      3637              910                    24.07
    4      3638              909                    24.09
    5      3638              909                    24.09
Every row validated once; no normalised-group overlap within any fold.


### 6. Inspect one fold without training a classifier
This demonstration fits each representation on just the first fold's fitting rows, then transforms its validation rows. We report aggregate dimensions and sparsity, not private message excerpts or corpus vocabulary. Unknown terms are ignored during transform. Empty feature rows can occur for punctuation-only text or text entirely outside the vocabulary; an app should warn when there are no usable text features.

In [7]:
fit_idx, valid_idx = folds[0]
feature_summary = []
for candidate in CANDIDATES:
    vectorizer = make_vectorizer(candidate)
    fit_features = vectorizer.fit_transform(X.iloc[fit_idx])
    valid_features = vectorizer.transform(X.iloc[valid_idx])
    assert fit_features.shape[1] == valid_features.shape[1]
    assert np.isfinite(fit_features.data).all() and (fit_features.data >= 0).all()
    assert np.isfinite(valid_features.data).all()
    empty_valid = int(np.count_nonzero(valid_features.getnnz(axis=1) == 0))
    feature_summary.append({"candidate": candidate, "features": fit_features.shape[1],
                            "fit_nonzeros": fit_features.nnz,
                            "fit_density_percent": round(100 * fit_features.nnz / np.prod(fit_features.shape), 3),
                            "empty_validation_rows": empty_valid})
    print(candidate, "punctuation-only input nonzeros:", vectorizer.transform(["!!!"]).nnz)
print(pd.DataFrame(feature_summary).to_string(index=False))
print("No classifier fitted and no candidate selected from these descriptive counts.")

word_tfidf punctuation-only input nonzeros: 0
word_tfidf_stopwords punctuation-only input nonzeros: 0
           candidate  features  fit_nonzeros  fit_density_percent  empty_validation_rows
          word_tfidf     30000        787038                0.721                      0
word_tfidf_stopwords     30000        454803                0.417                      0
No classifier fitted and no candidate selected from these descriptive counts.


### 7. Save the experiment plan
Save settings and aggregate diagnostics. The demonstration vocabulary is intentionally not exported as a production artifact. Notebook 4 must recreate and refit pipelines inside group-aware cross-validation; choose by spam-class F1, report precision, recall and average precision, and compare against a majority-class baseline.

Predicted probabilities, if later displayed, should be described as model estimates. They are not automatically calibrated confidence or proof that an email is safe.

In [8]:
plan = {
    "author": "Tinashe Clifford Makandise",
    "train_file_sha256": train_hash,
    "vectorizer_settings": VECTORIZER_SETTINGS,
    "candidates": CANDIDATES,
    "models": ["naive_bayes", "logistic_regression"],
    "validation": {"method": "StratifiedGroupKFold", "n_splits": 5, "shuffle": True, "random_state": 42},
    "group_column": "group_id", "primary_metric": "spam-class F1",
    "test_policy": "Do not open until final evaluation after model selection",
}
(REPORTS / "feature_engineering_plan.json").write_text(json.dumps(plan, indent=2), encoding="utf-8")
pd.DataFrame(fold_summary).to_csv(REPORTS / "training_cv_fold_summary.csv", index=False)
pd.DataFrame(feature_summary).to_csv(REPORTS / "tfidf_fold_diagnostics.csv", index=False)
assert hashlib.sha256(train_path.read_bytes()).hexdigest() == train_hash
print("Saved experiment plan and aggregate diagnostics. Original training CSV unchanged.")

Saved experiment plan and aggregate diagnostics. Original training CSV unchanged.


### Review questions
1. Why must TF-IDF be fitted inside each cross-validation fold?
2. What does a bigram add compared with individual words?
3. Why might stopword removal hurt a classifier?
4. Why must repeated normalised messages share a fold?
5. Why is a high spam score not proof of phishing?

**Next: Notebook 4 — Model Training and Comparison.** Train a majority baseline, Multinomial Naive Bayes and Logistic Regression using these pipelines and group-aware validation. Keep the final test partition reserved.